# LemurTF_IDF Experiment Summary

## Query ID Alignment Issue

During the initial retrieval experiments, the query IDs in `cran.qry` were found to be inconsistent with the query IDs used in the relevance judgments (`cranqrel`).

The numbers appearing after the `.I` tag in `cran.qry` are not sequential. For example:

- `.I 001`
- `.I 002`
- `.I 004`
- `.I 008`
- ...

However, the qrels in our experimental setup use sequential query IDs according to the position of the query blocks.

Therefore, the queries were reassigned sequential IDs based on their order in `cran.qry`:

1st query → qid 1  
2nd query → qid 2  
3rd query → qid 3  
...  
225th query → qid 225

The original number appearing after the `.I` tag was ignored.

This correction ensures that each retrieved result is evaluated against the correct relevance judgments.

## Important Clarification

The retrieval score produced by LemurTF_IDF and the relevance labels present in the qrels represent different quantities.

- **LemurTF_IDF score:** used to rank retrieved documents.
- **Qrel relevance label:** human-provided ground-truth relevance judgment.

Therefore, the LemurTF_IDF scores are not expected to numerically match the qrel relevance labels.

## Default LemurTF_IDF Result

After correcting the query ID alignment, the default LemurTF_IDF model was evaluated using the Cranfield collection.

The default Terrier parameters are:

- `k1 = 1.2`
- `b = 0.75`

The obtained results were:

**MAP = 0.294497**

**MRR = 0.512934**

**P@5 = 0.297778**

**P@10 = 0.227556**

**nDCG@10 = 0.323470**

The default LemurTF_IDF model therefore provides a reasonable sparse lexical retrieval baseline.

## Parameter Fine-Tuning

LemurTF_IDF contains two important tunable parameters:

### k1

`k1` controls term-frequency saturation.

A larger value of `k1` allows repeated occurrences of a query term within a document to have a greater influence on the retrieval score.

### b

`b` controls document-length normalization.

- `b = 0` means no document-length normalization.
- `b = 1` means full document-length normalization.

A grid search was performed using:

`k1 = {0.4, 0.8, 1.2, 1.6, 2.0}`

and

`b = {0.0, 0.25, 0.5, 0.75, 1.0}`

This produced 25 different parameter combinations.

The configurations were evaluated using MAP, MRR, P@5, P@10, and nDCG@10.

## Best Coarse-Grid Configuration

The best configuration according to MAP was:

**k1 = 2.0**

**b = 0.25**

The corresponding evaluation results were:

**MAP = 0.301619**

**MRR = 0.518089**

**P@5 = 0.304889**

**P@10 = 0.229778**

**nDCG@10 = 0.332116**

## Effect of Parameter Tuning

Compared with the default LemurTF_IDF configuration:

| Configuration | MAP | MRR | P@5 | P@10 | nDCG@10 |
|---|---:|---:|---:|---:|---:|
| Default (`k1=1.2`, `b=0.75`) | 0.294497 | 0.512934 | 0.297778 | 0.227556 | 0.323470 |
| Tuned (`k1=2.0`, `b=0.25`) | 0.301619 | 0.518089 | 0.304889 | 0.229778 | 0.332116 |

Parameter tuning improved all five evaluation measures.

The increase in `k1` indicates that allowing term frequency to have a stronger influence was beneficial for the Cranfield collection.

The decrease in `b` from 0.75 to 0.25 indicates that weaker document-length normalization produced better retrieval effectiveness for this dataset.

## Comparison with Other Sparse Models

The following models were also compared using the same preprocessing, queries, qrels, and evaluation measures:

| Model | MAP | MRR | P@5 | P@10 | nDCG@10 |
|---|---:|---:|---:|---:|---:|
| TF | 0.193439 | 0.429683 | 0.194667 | 0.151556 | 0.223466 |
| TF_IDF | 0.318897 | 0.554530 | 0.328889 | 0.236444 | 0.346963 |
| LemurTF_IDF Default | 0.294497 | 0.512934 | 0.297778 | 0.227556 | 0.323470 |
| LemurTF_IDF Tuned | 0.301619 | 0.518089 | 0.304889 | 0.229778 | 0.332116 |

The tuned LemurTF_IDF model performs substantially better than the simple TF baseline and improves over the default LemurTF_IDF configuration.

However, the standard Terrier TF_IDF model still achieves the highest effectiveness among these configurations.

## Final Finding

The LemurTF_IDF model was successfully implemented and tuned on the Cranfield collection.

Correct query-to-qrels alignment was essential for valid evaluation.

Parameter tuning improved LemurTF_IDF from:

**MAP = 0.294497**

to:

**MAP = 0.301619**

using:

**k1 = 2.0, b = 0.25**

The experiment shows that retrieval effectiveness is sensitive to term-frequency saturation and document-length normalization.

Since the best `k1` value found in the coarse grid lies at the upper boundary of the tested range, an additional fine-grained search around and above `k1 = 2.0` can be performed before selecting the final configuration.



Environment Setup

In [ ]:
%pip install -q python-terrier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.1/223.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 8.7 MB/s eta 0:00:00


In [ ]:
import pyterrier as pt
import pandas as pd
import re
import time

if not pt.started():
    pt.init()

print("PyTerrier version:", pt.__version__)

/tmp/ipykernel_2021/1066108698.py:6: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done
PyTerrier version: 1.1.2


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_2021/1066108698.py:7: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


Obtain Cranfield collection

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cran.tar.gz to cran.tar.gz


In [ ]:
import tarfile
with tarfile.open("cran.tar.gz", "r:gz") as cran:
  cran.extractall("cranfield")

/tmp/ipykernel_2021/1663252741.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  cran.extractall("cranfield")


Preprocessing of Documents

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving stopwords.txt to stopwords.txt
Saving outliers_preprocess.py to outliers_preprocess.py
Saving outliers_porter.py to outliers_porter.py


In [ ]:
import sys

sys.path.append("/content")

In [ ]:
from outliers_preprocess import (
    parse_documents,
    preprocess_text,
    load_stemmed_stopwords
)

from outliers_porter import PorterStemmer

In [ ]:
from pathlib import Path

input_path = Path("/content/cranfield/cran.all.1400")
stopwords_path = Path("/content/stopwords.txt")

In [ ]:
stemmer = PorterStemmer()
stemmed_stopwords = load_stemmed_stopwords(stopwords_path, stemmer)
documents = parse_documents(input_path)

In [ ]:
processed_documents = []

for doc_id, text in documents:
    tokens = preprocess_text(text, stemmer, stemmed_stopwords)

    processed_documents.append({
        "docno": str(doc_id),
        "text": " ".join(tokens)
    })

In [ ]:
docs = pd.DataFrame(processed_documents)
print(docs.head())

  docno                                               text
0     1  experiment investig aerodynam wing slipstream ...
1     2  simpl shear flow past flat plate incompress fl...
2     3  boundari layer simpl shear flow past flat plat...
3     4  approxim solut incompress laminar boundari lay...
4     5  dimension transient heat conduct doubl layer s...


Preprocessing of Queries

In [ ]:
def parse_queries(file_path):

    queries = []

    with open(file_path, "r", encoding="ascii", errors="ignore") as f:
        lines = f.readlines()

    current_query = []
    in_query = False

    for line in lines:

        line = line.rstrip("\n")

        # Start of a new query block
        if re.match(r"\.I\s+\d+", line):

            # Save previous query
            if current_query:
                queries.append({
                    "qid": str(len(queries) + 1),
                    "query": " ".join(current_query).strip()
                })

            current_query = []
            in_query = False

        elif line.strip() == ".W":

            in_query = True

        elif in_query:

            current_query.append(line.strip())

    # Save last query
    if current_query:
        queries.append({
            "qid": str(len(queries) + 1),
            "query": " ".join(current_query).strip()
        })

    return pd.DataFrame(queries)

In [ ]:
query = parse_queries("/content/cranfield/cran.qry")

In [ ]:
processed_queries = []

for _, row in query.iterrows():

    tokens = preprocess_text(row["query"],stemmer,stemmed_stopwords)

    processed_queries.append({
        "qid": str(row["qid"]),
        "query": " ".join(tokens)
    })

query = pd.DataFrame(processed_queries)

In [ ]:
display(query.head(10))

,qid,query
0,1,similar law obey construct aeroelast model hea...
1,2,structur aeroelast problem associ flight high ...
2,3,problem heat conduct composit slab solv far
3,4,criterion develop empir valid flow solut chemi...
4,5,chemic kinet applic hyperson aerodynam problem
5,6,theoret experiment guid turbul couett flow beh...
6,7,possibl relat avail pressur distribut ogiv for...
7,8,method dash exact approxim dash present avail ...
8,9,paper intern slip flow heat transfer studi
9,10,real ga transport properti air avail wide rang...


Indexing

In [ ]:
index_path = "/content/cranfield_lemur_tfidf_index"

indexer = pt.index.IterDictIndexer(
    index_path,
    meta=["docno"],
    text_attrs=["text"],
    overwrite=True
)

In [ ]:
start_time = time.perf_counter()

indexref = indexer.index(
    docs.to_dict("records")
)

index_time = time.perf_counter() - start_time

print(f"Indexing time: {index_time:.4f} seconds")

20:09:11.232 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (471) - further warnings are suppressed
20:09:12.945 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
Indexing time: 3.8593 seconds


In [ ]:
index = pt.IndexFactory.of(indexref)

In [ ]:
print(index.getCollectionStatistics())

Number of documents: 1400
Number of terms: 4389
Number of postings: 77353
Number of fields: 0
Number of tokens: 132181
Field names: []
Positions:   false



In [ ]:
print(index.getLexicon())

<org.terrier.structures.Lexicon at 0x7a7e387cb750 jclass=org/terrier/structures/Lexicon jself=<LocalRef obj=0xd121b82 at 0x7a7e3871f1d0>>


TF_IDF

In [ ]:
lemur_tfidf = pt.terrier.Retriever(
    index,
    wmodel="LemurTF_IDF"
)

In [ ]:
start = time.perf_counter()

lemur_results = lemur_tfidf.transform(query)

search_time = time.perf_counter() - start

print(f"Search time: {search_time:.4f} seconds")

Search time: 7.0538 seconds


In [ ]:
lemur_results

,qid,query,docid,docno,rank,score
0,1,similar law obey construct aeroelast model hea...,183,184,0,65.869580
1,1,similar law obey construct aeroelast model hea...,485,486,1,64.612539
2,1,similar law obey construct aeroelast model hea...,50,51,2,62.776695
3,1,similar law obey construct aeroelast model hea...,11,12,3,60.684798
4,1,similar law obey construct aeroelast model hea...,572,573,4,59.258304
...,...,...,...,...,...,...
188970,225,design factor control lift drag ratio mach num...,836,837,891,0.733339
188971,225,design factor control lift drag ratio mach num...,1143,1144,892,0.715901
188972,225,design factor control lift drag ratio mach num...,82,83,893,0.713477
188973,225,design factor control lift drag ratio mach num...,1391,1392,894,0.703944


Evaluation

In [ ]:
qrels = pd.read_csv(
    "/content/cranfield/cranqrel",
    sep=r"\s+",
    header=None,
    names=["qid", "docno", "label"]
)

qrels["qid"] = qrels["qid"].astype(str)
qrels["docno"] = qrels["docno"].astype(str)

print(qrels.head())
print("Number of relevance judgments:", len(qrels))
print("Number of queries:", qrels["qid"].nunique())

  qid docno  label
0   1   184      2
1   1    29      2
2   1    31      2
3   1    12      3
4   1    51      3
Number of relevance judgments: 1837
Number of queries: 225


In [ ]:
print("LemurTF-IDF columns:")
print(lemur_results.columns.tolist())

print("\nQrels columns:")
print(qrels.columns.tolist())

LemurTF-IDF columns:
['qid', 'query', 'docid', 'docno', 'rank', 'score']

Qrels columns:
['qid', 'docno', 'label']


In [ ]:
print(lemur_results[["qid", "docno"]].dtypes)
print(qrels[["qid", "docno"]].dtypes)

qid      object
docno    object
dtype: object
qid      object
docno    object
dtype: object


In [ ]:
lemur_eval = pt.Evaluate(
    lemur_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

In [ ]:
lemur_results_table = pd.DataFrame([{
    "Model": "LemurTF-IDF",
    "MAP": lemur_eval["map"],
    "MRR": lemur_eval["recip_rank"],
    "P@5": lemur_eval["P.5"],
    "P@10": lemur_eval["P.10"],
    "nDCG@10": lemur_eval["ndcg_cut.10"],
    "Index Time (s)": index_time,
    "Search Time (s)": search_time
}])

lemur_results_table

,Model,MAP,MRR,P@5,P@10,nDCG@10,Index Time (s),Search Time (s)
0,LemurTF-IDF,0.294497,0.512934,0.297778,0.227556,0.32347,3.859258,7.053797


In [ ]:
lemur_pairs = set(
    zip(lemur_results["qid"], lemur_results["docno"])
)

qrel_pairs = set(
    zip(qrels["qid"], qrels["docno"])
)

overlap = lemur_pairs & qrel_pairs

print("TF-IDF pairs:", len(lemur_pairs))
print("Qrels pairs:", len(qrel_pairs))
print("Matching pairs:", len(overlap))

TF-IDF pairs: 188975
Qrels pairs: 1837
Matching pairs: 1746


In [ ]:
print("LemurTF-IDF queries:", lemur_results["qid"].nunique())
print("Qrel queries:", qrels["qid"].nunique())

LemurTF-IDF queries: 225
Qrel queries: 225


In [ ]:
print("LemurTF-IDF top 20 for query 1")

display(
    lemur_results[
        lemur_results["qid"] == "1"
    ][
        ["qid", "docno", "rank", "score"]
    ].head(20)
)

LemurTF-IDF top 20 for query 1


,qid,docno,rank,score
0,1,184,0,65.869580
1,1,486,1,64.612539
2,1,51,2,62.776695
3,1,12,3,60.684798
4,1,573,4,59.258304
5,1,944,5,57.189316
6,1,746,6,49.838153
7,1,878,7,48.507813
8,1,665,8,46.928869
9,1,875,9,45.550509


In [ ]:
print(lemur_results["qid"].unique()[:20])
print(qrels["qid"].unique()[:20])

['1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15' '16'
 '17' '18' '19' '20']
['1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15' '16'
 '17' '18' '19' '20']


In [ ]:
print(query["qid"].unique()[:30])

['1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15' '16'
 '17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30']


In [ ]:
print("Number of queries:", query["qid"].nunique())
print("Number of qrel queries:", qrels["qid"].nunique())

print("\nQueries in qrels but NOT in query file:")
print(sorted(
    set(qrels["qid"]) - set(query["qid"]),
    key=int
))

Number of queries: 225
Number of qrel queries: 225

Queries in qrels but NOT in query file:
[]


In [ ]:
display(qrels[qrels["qid"] == "1"])

,qid,docno,label
0,1,184,2
1,1,29,2
2,1,31,2
3,1,12,3
4,1,51,3
5,1,102,3
6,1,13,4
7,1,14,4
8,1,15,4
9,1,57,2


In [ ]:
display(
    lemur_results[
        lemur_results["qid"] == "1"
    ][["qid", "docno", "rank", "score"]].head(20)
)

,qid,docno,rank,score
0,1,184,0,65.869580
1,1,486,1,64.612539
2,1,51,2,62.776695
3,1,12,3,60.684798
4,1,573,4,59.258304
5,1,944,5,57.189316
6,1,746,6,49.838153
7,1,878,7,48.507813
8,1,665,8,46.928869
9,1,875,9,45.550509


In [ ]:
print(qrels["label"].unique())

[ 2  3  4 -1  1]


Fine Tuning LemurTF_IDF

*Find Parameters*

In [ ]:
help(pt.terrier.Retriever)

Help on class Retriever in module pyterrier.terrier.retriever:

class Retriever(pyterrier.transformer.Transformer)
 |  Retriever(
 |      index_location: Union[str, Any],
 |      controls: Optional[Dict[str, str]] = None,
 |      properties: Optional[Dict[str, str]] = None,
 |      metadata: List[str] = ['docno'],
 |      num_results: Optional[int] = None,
 |      wmodel: Union[str, Callable, NoneType] = None,
 |      tokeniser: Union[str, pyterrier.terrier.tokeniser.TerrierTokeniser] = <TerrierTokeniser.english: 'english'>,
 |      threads: int = 1,
 |      verbose: bool = False
 |  )
 |
 |  Use this class for retrieval by Terrier
 |
 |  Method resolution order:
 |      Retriever
 |      pyterrier.transformer.Transformer
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __eq__(self, other)
 |      Return self==value.
 |
 |  __getstate__(self)
 |      Helper for pickle.
 |
 |  __hash__(self)
 |      Return hash(self).
 |
 |  __init__(
 |      self,
 |      index_location: Uni

In [ ]:
print(pt.terrier.Retriever.__doc__)
import pyterrier.terrier as pt_terrier
import inspect

print(inspect.getsource(pt.terrier.Retriever))


Use this class for retrieval by Terrier

@pt.java.required
class Retriever(pt.Transformer):
    """
    Use this class for retrieval by Terrier
    """

    @staticmethod
    def matchop(t, w=1):
        """
        Static method used for rewriting a query term to use a MatchOp operator if it contains
        anything except ASCII letters or digits.
        """
        import base64
        import string
        if not all(a in string.ascii_letters + string.digits for a in t):
            encoded = base64.b64encode(t.encode('utf-8')).decode("utf-8") 
            t = f'#base64({encoded})'
        if w != 1:
            t = f'#combine:0={w}({t})'
        return t


    @staticmethod
    def from_dataset(dataset : Union[str,Dataset], 
            variant : Optional[str] = None, 
            version='latest',            
            **kwargs):
        """
        Static method that instantiates a Retriever object from a pre-built index access via a dataset.
        Pre-built indices are o

In [ ]:
import pyterrier as pt
if not pt.java.started():
    pt.java.init()

from jnius import autoclass

# Load and instantiate the Java class
ModelClass = autoclass("org.terrier.matching.models.LemurTF_IDF")
instance = ModelClass()

# Get the real java.lang.Class object via the instance, THEN reflect on it
java_class = instance.getClass()

print("Methods:")
for method in java_class.getDeclaredMethods():
    print(" ", method.getName())

print("\nFields:")
for field in java_class.getDeclaredFields():
    print(" ", field.getName())

Methods:
  score
  prepare
  getInfo

Fields:
  serialVersionUID
  k_1
  b


In [ ]:
k1_values = [0.4, 0.8, 1.2, 1.6, 2.0]
b_values = [0.0, 0.25, 0.5, 0.75, 1.0]

tuning_results = []

for k1 in k1_values:

    for b in b_values:

        model = pt.terrier.Retriever(
            index,
            wmodel="LemurTF_IDF",
            controls={
                "LemurTF_IDF.k_1": str(k1),
                "LemurTF_IDF.b": str(b)
            }
        )

        start = time.perf_counter()

        run = model.transform(query)

        current_search_time = time.perf_counter() - start

        evaluation = pt.Evaluate(
            run,
            qrels,
            metrics=[
                "map",
                "recip_rank",
                "P.5",
                "P.10",
                "ndcg_cut.10"
            ]
        )

        tuning_results.append({
            "k1": k1,
            "b": b,
            "MAP": evaluation["map"],
            "MRR": evaluation["recip_rank"],
            "P@5": evaluation["P.5"],
            "P@10": evaluation["P.10"],
            "nDCG@10": evaluation["ndcg_cut.10"],
            "Search Time (s)": current_search_time
        })


tuning_df = pd.DataFrame(tuning_results)

tuning_df = tuning_df.sort_values(
    by="MAP",
    ascending=False
).reset_index(drop=True)

display(tuning_df)

,k1,b,MAP,MRR,P@5,P@10,nDCG@10,Search Time (s)
0,2.0,0.25,0.301619,0.518089,0.304889,0.229778,0.332116,4.956114
1,2.0,0.50,0.299145,0.508978,0.314667,0.234222,0.329391,4.555601
2,1.6,0.25,0.298951,0.517998,0.299556,0.226222,0.328560,5.395701
3,2.0,0.75,0.298837,0.514541,0.309333,0.232000,0.327443,5.862100
4,2.0,1.00,0.298520,0.514678,0.309333,0.226222,0.324423,4.403579
5,2.0,0.00,0.297374,0.539316,0.295111,0.218667,0.329442,5.041389
6,1.6,0.75,0.297235,0.510387,0.302222,0.232000,0.327380,5.878691
7,1.6,0.00,0.296619,0.541936,0.293333,0.218667,0.329030,4.526334
8,1.6,0.50,0.296525,0.508374,0.305778,0.232444,0.328202,4.462006
9,1.6,1.00,0.296241,0.512497,0.303111,0.224444,0.322569,4.415133


In [ ]:
best = tuning_df.iloc[0]

print("Best LemurTF_IDF configuration")
print("-------------------------------")
print("k1       =", best["k1"])
print("b        =", best["b"])
print("MAP      =", best["MAP"])
print("MRR      =", best["MRR"])
print("P@5      =", best["P@5"])
print("P@10     =", best["P@10"])
print("nDCG@10  =", best["nDCG@10"])

Best LemurTF_IDF configuration
-------------------------------
k1       = 2.0
b        = 0.25
MAP      = 0.301618974194225
MRR      = 0.5180887034257201
P@5      = 0.30488888888888904
P@10     = 0.22977777777777805
nDCG@10  = 0.3321155623623562


In [ ]:
best_k1 = float(best["k1"])
best_b = float(best["b"])

tuned_lemur = pt.terrier.Retriever(
    index,
    wmodel="LemurTF_IDF",
    controls={
        "LemurTF_IDF.k_1": str(best_k1),
        "LemurTF_IDF.b": str(best_b)
    }
)

In [ ]:
comparison = pt.Experiment(
    [
        lemur_tfidf,
        tuned_lemur
    ],
    query,
    qrels,
    eval_metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ],
    names=[
        "LemurTF_IDF Default",
        f"LemurTF_IDF Tuned (k1={best_k1}, b={best_b})"
    ]
)

display(comparison)

,name,map,recip_rank,P.5,P.10,ndcg_cut.10
0,LemurTF_IDF Default,0.294497,0.512934,0.297778,0.227556,0.323470
1,"LemurTF_IDF Tuned (k1=2.0, b=0.25)",0.301619,0.518089,0.304889,0.229778,0.332116


Comparison on TF, TF IDF and Lemur TF IDF

In [ ]:
tf = pt.terrier.Retriever(
    index,
    wmodel="Tf"
)

tfidf = pt.terrier.Retriever(
    index,
    wmodel="TF_IDF"
)
systems = [
    tf,
    tfidf,
    lemur_tfidf,
    tuned_lemur
]

experiment = pt.Experiment(
    systems,
    query,
    qrels,
    eval_metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ],
    names=[
        "TF",
        "TF_IDF",
        "LemurTF_IDF Default",
        "LemurTF_IDF Tuned"
    ]
)

display(experiment)



,name,map,recip_rank,P.5,P.10,ndcg_cut.10
0,TF,0.193439,0.429683,0.194667,0.151556,0.223466
1,TF_IDF,0.318897,0.554530,0.328889,0.236444,0.346963
2,LemurTF_IDF Default,0.294497,0.512934,0.297778,0.227556,0.323470
3,LemurTF_IDF Tuned,0.301619,0.518089,0.304889,0.229778,0.332116


## Relevance Feedback and Query Expansion (Rocchio, Vector Space Model)

The sections above tuned **term-weighting** for a single vector space model (`LemurTF_IDF`). This section adds a genuinely different technique on top of the retrieval pipeline: **relevance feedback / query expansion** using the classic **Rocchio algorithm** (Rocchio, 1971), which stays entirely inside the sparse vector space model family (TF-IDF vectors + cosine similarity). No probabilistic (BM25/DFR) or dense/neural model is introduced anywhere.

**How it works (blind / pseudo-relevance feedback):**

1. Run an initial ranking with a base retriever (the tuned `LemurTF_IDF` model).
2. Assume the top `num_fb_docs` documents are relevant (*pseudo*-relevance feedback — the true `qrels` labels are never used to build the expanded query, so evaluating on the same `qrels` afterwards stays fair).
3. Represent the query and all documents as TF-IDF vectors (`TfidfVectorizer`, fit on the same preprocessed/stemmed text used for indexing).
4. Move the query vector toward the centroid of the feedback documents:

   `q_new = alpha * q_orig + beta * mean(TF-IDF vectors of feedback docs) - gamma * mean(TF-IDF vectors of non-relevant docs)`

   (`gamma` defaults to 0, i.e. no negative feedback — standard for blind PRF.)
5. Re-rank all documents by cosine similarity against `q_new`.

This is wrapped as a proper `pt.Transformer`, so it can sit inside `pt.Experiment` next to the existing `tf`, `tfidf`, `lemur_tfidf` and `tuned_lemur` systems for a direct, apples-to-apples comparison.

In [ ]:
%pip install -q scikit-learn scipy

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import scipy.sparse as sp

In [ ]:
class RocchioQE(pt.Transformer):
    """
    Pseudo-relevance feedback / query expansion using the classic Rocchio
    algorithm, operating purely on sparse TF-IDF vectors (vector space model).

    q_new = alpha * q_orig + beta * mean(TF-IDF vectors of top num_fb_docs
            feedback documents) - gamma * mean(TF-IDF vectors of bottom
            feedback documents, optional negative feedback)

    If `expansion_terms` (K) is set, only the K highest-weight terms of the
    feedback centroid(s) are kept before being added to the query ("top-K
    term selection") -- this is the standard way real query-expansion
    systems avoid dragging in hundreds of low-weight, noisy terms from the
    feedback documents. expansion_terms=None keeps the full centroid
    (original, unpruned behaviour).

    Parameters
    ----------
    docs_df : pd.DataFrame
        Preprocessed documents with columns ["docno", "text"].
    base_retriever : pt.Transformer
        Any PyTerrier retriever (e.g. tuned_lemur) used ONLY to obtain the
        initial ranking that feedback documents are drawn from.
    num_fb_docs : int
        Number of top-ranked documents assumed relevant (blind feedback).
    alpha, beta, gamma : float
        Rocchio weights for original query, positive feedback and negative
        feedback respectively. gamma=0 disables negative feedback.
    expansion_terms : int or None
        Keep only the top-K highest-weight terms of each feedback centroid.
        None = no pruning (use the full centroid).
    top_k : int
        Number of documents to return per query after re-ranking.
    """

    def __init__(self, docs_df, base_retriever, num_fb_docs=10,
                 alpha=1.0, beta=0.75, gamma=0.0, expansion_terms=None,
                 top_k=1000):
        self.docs_df = docs_df.reset_index(drop=True)
        self.base_retriever = base_retriever
        self.num_fb_docs = num_fb_docs
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.expansion_terms = expansion_terms
        self.top_k = top_k

        self.vectorizer = TfidfVectorizer()
        self.doc_matrix = self.vectorizer.fit_transform(self.docs_df["text"])
        self.docno_to_idx = {
            docno: i for i, docno in enumerate(self.docs_df["docno"])
        }

    @staticmethod
    def _select_top_k(vec, k):
        """Keep only the k highest-weight entries of a 1-row sparse vector."""
        vec = sp.csr_matrix(vec)
        if vec.nnz <= k:
            return vec

        data = vec.data
        indices = vec.indices
        indptr = vec.indptr

        top_local_idx = np.argpartition(-data, k)[:k]
        mask = np.zeros(len(data), dtype=bool)
        mask[top_local_idx] = True

        new_data = np.where(mask, data, 0.0)
        pruned = sp.csr_matrix((new_data, indices, indptr), shape=vec.shape)
        pruned.eliminate_zeros()
        return pruned

    def transform(self, topics):
        init_res = self.base_retriever.transform(topics)

        all_rows = []

        for qid, qgroup in topics.groupby("qid"):
            query_text = qgroup.iloc[0]["query"]
            q_vec = self.vectorizer.transform([query_text])

            q_res = init_res[init_res["qid"] == qid].sort_values("rank")

            fb_docnos = q_res["docno"].head(self.num_fb_docs).tolist()
            fb_idxs = [
                self.docno_to_idx[d] for d in fb_docnos if d in self.docno_to_idx
            ]

            if fb_idxs:
                pos_centroid = sp.csr_matrix(
                    self.doc_matrix[fb_idxs].mean(axis=0)
                )
            else:
                pos_centroid = sp.csr_matrix(q_vec.shape)

            if self.gamma > 0:
                nonrel_docnos = q_res["docno"].tail(self.num_fb_docs).tolist()
                nonrel_idxs = [
                    self.docno_to_idx[d] for d in nonrel_docnos if d in self.docno_to_idx
                ]
                if nonrel_idxs:
                    neg_centroid = sp.csr_matrix(
                        self.doc_matrix[nonrel_idxs].mean(axis=0)
                    )
                else:
                    neg_centroid = sp.csr_matrix(q_vec.shape)
            else:
                neg_centroid = sp.csr_matrix(q_vec.shape)

            # --- Top-K term selection (query expansion pruning) ---
            if self.expansion_terms is not None:
                pos_centroid = self._select_top_k(pos_centroid, self.expansion_terms)
                if self.gamma > 0:
                    neg_centroid = self._select_top_k(neg_centroid, self.expansion_terms)

            new_q_vec = (
                self.alpha * q_vec
                + self.beta * pos_centroid
                - self.gamma * neg_centroid
            )
            new_q_vec = sp.csr_matrix(new_q_vec)
            new_q_vec.data[new_q_vec.data < 0] = 0
            new_q_vec.eliminate_zeros()

            scores = cosine_similarity(new_q_vec, self.doc_matrix).flatten()
            ranked_idx = np.argsort(-scores)[: self.top_k]

            for rank, idx in enumerate(ranked_idx):
                score = float(scores[idx])
                if score <= 0:
                    continue
                all_rows.append({
                    "qid": qid,
                    "docno": self.docs_df.iloc[idx]["docno"],
                    "rank": rank,
                    "score": score
                })

        return pd.DataFrame(all_rows)

### Build the Rocchio pipeline on top of the tuned LemurTF_IDF ranking

In [ ]:
start = time.perf_counter()

rocchio_qe = RocchioQE(
    docs_df=docs,
    base_retriever=tuned_lemur,
    num_fb_docs=10,
    alpha=1.0,
    beta=0.75,
    gamma=0.0,
    top_k=1000
)

rocchio_build_time = time.perf_counter() - start

print(f"Rocchio vector space build time: {rocchio_build_time:.4f} seconds")

Rocchio vector space build time: 0.5162 seconds


In [ ]:
start = time.perf_counter()

rocchio_results = rocchio_qe.transform(query)

rocchio_search_time = time.perf_counter() - start

print(f"Rocchio QE search time: {rocchio_search_time:.4f} seconds")

rocchio_results["qid"] = rocchio_results["qid"].astype(str)
rocchio_results["docno"] = rocchio_results["docno"].astype(str)

rocchio_eval = pt.Evaluate(
    rocchio_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

print(rocchio_eval)

Rocchio QE search time: 31.2543 seconds
{'map': 0.33702641690605784, 'recip_rank': 0.5496903184085151, 'P.5': 0.33955555555555567, 'P.10': 0.26044444444444464, 'ndcg_cut.10': 0.35974900329590626}


### Tune Rocchio parameters (`num_fb_docs`, `beta`)

In [ ]:
fb_docs_values = [3, 5, 10, 15, 20]
beta_values = [0.25, 0.5, 0.75, 1.0]

rocchio_tuning_results = []

for nfb in fb_docs_values:

    for beta_v in beta_values:

        rqe = RocchioQE(
            docs_df=docs,
            base_retriever=tuned_lemur,
            num_fb_docs=nfb,
            alpha=1.0,
            beta=beta_v,
            gamma=0.0,
            top_k=1000
        )

        start = time.perf_counter()

        run = rqe.transform(query)

        current_search_time = time.perf_counter() - start

        run["qid"] = run["qid"].astype(str)
        run["docno"] = run["docno"].astype(str)

        evaluation = pt.Evaluate(
            run,
            qrels,
            metrics=[
                "map",
                "recip_rank",
                "P.5",
                "P.10",
                "ndcg_cut.10"
            ]
        )

        rocchio_tuning_results.append({
            "num_fb_docs": nfb,
            "beta": beta_v,
            "MAP": evaluation["map"],
            "MRR": evaluation["recip_rank"],
            "P@5": evaluation["P.5"],
            "P@10": evaluation["P.10"],
            "nDCG@10": evaluation["ndcg_cut.10"],
            "Search Time (s)": current_search_time
        })


rocchio_tuning_df = pd.DataFrame(rocchio_tuning_results)

rocchio_tuning_df = rocchio_tuning_df.sort_values(
    by="MAP",
    ascending=False
).reset_index(drop=True)

display(rocchio_tuning_df)

,num_fb_docs,beta,MAP,MRR,P@5,P@10,nDCG@10,Search Time (s)
0,3,1.00,0.340437,0.527942,0.359111,0.270667,0.366833,16.554924
1,10,1.00,0.340174,0.554886,0.340444,0.261778,0.361868,16.687338
2,5,1.00,0.339427,0.546567,0.334222,0.264889,0.362829,17.373414
3,5,0.75,0.338422,0.549377,0.344889,0.264889,0.363180,16.472985
4,3,0.75,0.337406,0.528078,0.356444,0.268000,0.363928,16.661889
5,10,0.75,0.337026,0.549690,0.339556,0.260444,0.359749,18.098307
6,3,0.50,0.336598,0.541829,0.357333,0.262667,0.362318,17.372777
7,15,1.00,0.335067,0.554392,0.342222,0.257333,0.355272,16.633387
8,5,0.50,0.334512,0.548947,0.345778,0.260000,0.357610,16.667957
9,15,0.75,0.333562,0.551511,0.337778,0.256444,0.355710,16.743111


In [ ]:
best_rocchio = rocchio_tuning_df.iloc[0]

print("Best Rocchio configuration")
print("---------------------------")
print("num_fb_docs =", best_rocchio["num_fb_docs"])
print("beta        =", best_rocchio["beta"])
print("MAP         =", best_rocchio["MAP"])
print("MRR         =", best_rocchio["MRR"])
print("P@5         =", best_rocchio["P@5"])
print("P@10        =", best_rocchio["P@10"])
print("nDCG@10     =", best_rocchio["nDCG@10"])

Best Rocchio configuration
---------------------------
num_fb_docs = 3.0
beta        = 1.0
MAP         = 0.34043700394354004
MRR         = 0.527942498888748
P@5         = 0.3591111111111113
P@10        = 0.27066666666666683
nDCG@10     = 0.36683265197352866


In [ ]:
start = time.perf_counter()

best_rocchio_qe = RocchioQE(
    docs_df=docs,
    base_retriever=tuned_lemur,
    num_fb_docs=int(best_rocchio["num_fb_docs"]),
    alpha=1.0,
    beta=float(best_rocchio["beta"]),
    gamma=0.0,
    top_k=1000
)

rocchio_build_time = time.perf_counter() - start

start = time.perf_counter()

best_rocchio_results = best_rocchio_qe.transform(query)

best_rocchio_search_time = time.perf_counter() - start

print(f"Best Rocchio build time: {rocchio_build_time:.4f} seconds")
print(f"Best Rocchio search time: {best_rocchio_search_time:.4f} seconds")

Best Rocchio build time: 0.1210 seconds
Best Rocchio search time: 16.5994 seconds


### Final comparison: baseline vector space models vs. Rocchio relevance feedback / query expansion

In [ ]:
start = time.perf_counter()
tf_results = tf.transform(query)
tf_search_time = time.perf_counter() - start

start = time.perf_counter()
tfidf_results = tfidf.transform(query)
tfidf_search_time = time.perf_counter() - start

start = time.perf_counter()
tuned_lemur_results = tuned_lemur.transform(query)
tuned_lemur_search_time = time.perf_counter() - start

final_systems = [
    tf,
    tfidf,
    lemur_tfidf,
    tuned_lemur,
    best_rocchio_qe
]

rocchio_name = (
    f"Rocchio RF/QE (fb_docs={int(best_rocchio['num_fb_docs'])}, "
    f"beta={best_rocchio['beta']})"
)

final_names = [
    "TF",
    "TF_IDF",
    "LemurTF_IDF Default",
    "LemurTF_IDF Tuned",
    rocchio_name
]

final_experiment = pt.Experiment(
    final_systems,
    query,
    qrels,
    eval_metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ],
    names=final_names
)

timing_summary = pd.DataFrame([
    {"name": "TF", "Index/Build Time (s)": index_time, "Search Time (s)": tf_search_time},
    {"name": "TF_IDF", "Index/Build Time (s)": index_time, "Search Time (s)": tfidf_search_time},
    {"name": "LemurTF_IDF Default", "Index/Build Time (s)": index_time, "Search Time (s)": search_time},
    {"name": "LemurTF_IDF Tuned", "Index/Build Time (s)": index_time, "Search Time (s)": tuned_lemur_search_time},
    {
        "name": rocchio_name,
        "Index/Build Time (s)": rocchio_build_time,
        "Search Time (s)": best_rocchio_search_time
    },
])

final_report = final_experiment.merge(timing_summary, on="name")

display(final_report)

/usr/local/lib/python3.13/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines do not produce the required outputs for evaluation ({'qid', 'docno', 'score'}):
 - Pipeline #4: Rocchio RF/QE (fb_docs=3, beta=1.0) (<__main__.RocchioQE object at 0x7a7de03b5910>). Produces []

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


,name,map,recip_rank,P.5,P.10,ndcg_cut.10,Index/Build Time (s),Search Time (s)
0,TF,0.193439,0.429683,0.194667,0.151556,0.223466,3.859258,4.445653
1,TF_IDF,0.318897,0.554530,0.328889,0.236444,0.346963,3.859258,5.022775
2,LemurTF_IDF Default,0.294497,0.512934,0.297778,0.227556,0.323470,3.859258,7.053797
3,LemurTF_IDF Tuned,0.301619,0.518089,0.304889,0.229778,0.332116,3.859258,5.309957
4,"Rocchio RF/QE (fb_docs=3, beta=1.0)",0.340437,0.527942,0.359111,0.270667,0.366833,0.120984,16.599401


### Notes for the report

- **LemurTF_IDF Tuned** re-uses the best `k1`/`b` found in the parameter-tuning grid search above.
- **Rocchio RF/QE** performs *blind* (pseudo) relevance feedback: it never looks at the true `qrels` labels, only at the top-ranked documents from `tuned_lemur`, so comparing it against the other rows on the same `qrels` is a fair, non-circular evaluation.
- All five systems are **sparse vector space models** — no probabilistic (BM25/DFR) and no dense/neural retrieval model is used anywhere in this notebook, satisfying the assignment constraint.
- `Index/Build Time (s)` is the one-off cost of building the underlying representation (the Terrier inverted index for the first four rows, the TF-IDF matrix for Rocchio); `Search Time (s)` is the time to answer all test queries.
- In the report, quote the winning `num_fb_docs`/`beta` combination from the tuning grid and discuss the trend (typically a moderate `beta` around 0.5-0.75 helps most, while a very large `num_fb_docs` pulls the query toward off-topic terms and hurts MAP/nDCG).

## Improvement 1: Top-K Term Selection for Rocchio Query Expansion

`RocchioQE` now supports `expansion_terms` (K): instead of adding the *entire* feedback centroid (which includes many low-weight, noisy terms) back into the query, only the K highest-weight terms are kept. This is tuned here on top of the best `num_fb_docs` / `beta` found earlier, holding them fixed.

In [ ]:
expansion_terms_values = [5, 10, 20, 40, 80, None]  # None = no pruning (previous behaviour)

topk_tuning_results = []

for k_terms in expansion_terms_values:

    rqe_k = RocchioQE(
        docs_df=docs,
        base_retriever=tuned_lemur,
        num_fb_docs=int(best_rocchio["num_fb_docs"]),
        alpha=1.0,
        beta=float(best_rocchio["beta"]),
        gamma=0.0,
        expansion_terms=k_terms,
        top_k=1000
    )

    start = time.perf_counter()

    run_k = rqe_k.transform(query)

    current_search_time = time.perf_counter() - start

    run_k["qid"] = run_k["qid"].astype(str)
    run_k["docno"] = run_k["docno"].astype(str)

    evaluation_k = pt.Evaluate(
        run_k,
        qrels,
        metrics=[
            "map",
            "recip_rank",
            "P.5",
            "P.10",
            "ndcg_cut.10"
        ]
    )

    topk_tuning_results.append({
        "expansion_terms": k_terms if k_terms is not None else "all (no pruning)",
        "MAP": evaluation_k["map"],
        "MRR": evaluation_k["recip_rank"],
        "P@5": evaluation_k["P.5"],
        "P@10": evaluation_k["P.10"],
        "nDCG@10": evaluation_k["ndcg_cut.10"],
        "Search Time (s)": current_search_time
    })


topk_tuning_df = pd.DataFrame(topk_tuning_results)

topk_tuning_df = topk_tuning_df.sort_values(
    by="MAP",
    ascending=False
).reset_index(drop=True)

display(topk_tuning_df)

,expansion_terms,MAP,MRR,P@5,P@10,nDCG@10,Search Time (s)
0,all (no pruning),0.340437,0.527942,0.359111,0.270667,0.366833,18.025349
1,80,0.339851,0.530272,0.355556,0.272444,0.367279,17.097868
2,40,0.339708,0.530209,0.354667,0.268889,0.366653,19.390141
3,20,0.338847,0.540851,0.356444,0.261333,0.361329,17.084085
4,10,0.331307,0.532034,0.344889,0.258222,0.359818,16.152083
5,5,0.318002,0.521362,0.331556,0.250667,0.348458,15.817692


In [ ]:
best_topk_row = topk_tuning_df.iloc[0]
best_expansion_terms = best_topk_row["expansion_terms"]
best_expansion_terms = None if best_expansion_terms == "all (no pruning)" else int(best_expansion_terms)

print("Best expansion_terms (K):", best_expansion_terms)
print("MAP with best K:", best_topk_row["MAP"])
print("MAP without pruning (baseline Rocchio):", rocchio_eval["map"])

Best expansion_terms (K): None
MAP with best K: 0.34043700394354004
MAP without pruning (baseline Rocchio): 0.33702641690605784


In [ ]:
start = time.perf_counter()

best_rocchio_qe = RocchioQE(
    docs_df=docs,
    base_retriever=tuned_lemur,
    num_fb_docs=int(best_rocchio["num_fb_docs"]),
    alpha=1.0,
    beta=float(best_rocchio["beta"]),
    gamma=0.0,
    expansion_terms=best_expansion_terms,
    top_k=1000
)

rocchio_build_time = time.perf_counter() - start

start = time.perf_counter()

best_rocchio_results = best_rocchio_qe.transform(query)

best_rocchio_search_time = time.perf_counter() - start

best_rocchio_results["qid"] = best_rocchio_results["qid"].astype(str)
best_rocchio_results["docno"] = best_rocchio_results["docno"].astype(str)

best_rocchio_eval = pt.Evaluate(
    best_rocchio_results,
    qrels,
    metrics=["map", "recip_rank", "P.5", "P.10", "ndcg_cut.10"]
)

print(f"Best Rocchio (fb_docs={int(best_rocchio['num_fb_docs'])}, "
      f"beta={best_rocchio['beta']}, K={best_expansion_terms}) build time: "
      f"{rocchio_build_time:.4f}s, search time: {best_rocchio_search_time:.4f}s")
print(best_rocchio_eval)

Best Rocchio (fb_docs=3, beta=1.0, K=None) build time: 0.1457s, search time: 16.5991s
{'map': 0.34043700394354004, 'recip_rank': 0.527942498888748, 'P.5': 0.3591111111111113, 'P.10': 0.27066666666666683, 'ndcg_cut.10': 0.36683265197352866}


## Improvement 2: Title Field Boosting

Cranfield documents have a `.T` (title) and `.W` (body/abstract) field. Titles are short and topically concentrated, so weighting title terms more heavily is a classic sparse-VSM improvement. The simplest way to do this that works uniformly with **every** weighting model already in this notebook (`Tf`, `TF_IDF`, `LemurTF_IDF`, and `RocchioQE`'s own `TfidfVectorizer`) is **term-frequency boosting by repetition**: the title's preprocessed tokens are repeated `boost_factor` times before being concatenated with the body tokens and (re-)indexed. This raises the TF (and therefore the TF-IDF weight) of title terms without needing a multi-field weighting model.

A fresh parser is used here (rather than the original `outliers_preprocess.parse_documents`) because it needs the `.T` and `.W` fields kept **separate** instead of merged into one field; it reuses the same `stemmer`, `stemmed_stopwords` and `preprocess_text` already loaded earlier, so preprocessing stays identical to the rest of the notebook.

In [ ]:
def parse_documents_with_fields(file_path):
    """Parse cran.all.1400, keeping the .T (title) and .W (body) fields separate."""

    with open(file_path, "r", encoding="ascii", errors="ignore") as f:
        lines = f.readlines()

    docs = []
    current = None
    section = None

    for line in lines:
        line = line.rstrip("\n")

        if re.match(r"\.I\s+\d+", line):
            if current is not None:
                docs.append(current)
            current = {"title": [], "body": []}
            section = None
            continue

        if line.startswith(".T"):
            section = "title"
            continue
        if line.startswith(".A"):
            section = "author"
            continue
        if line.startswith(".B"):
            section = "biblio"
            continue
        if line.startswith(".W"):
            section = "body"
            continue

        if current is None:
            continue

        if section == "title":
            current["title"].append(line)
        elif section == "body":
            current["body"].append(line)
        # author / bibliography fields are ignored

    if current is not None:
        docs.append(current)

    result = []
    for i, d in enumerate(docs):
        result.append({
            "docno": str(i + 1),
            "title": " ".join(d["title"]).strip(),
            "body": " ".join(d["body"]).strip()
        })

    return result

In [ ]:
raw_docs_fields = parse_documents_with_fields(input_path)

title_boost_processed = []

for d in raw_docs_fields:
    title_tokens = preprocess_text(d["title"], stemmer, stemmed_stopwords)
    body_tokens = preprocess_text(d["body"], stemmer, stemmed_stopwords)

    title_boost_processed.append({
        "docno": d["docno"],
        "title_tokens": title_tokens,
        "body_tokens": body_tokens
    })

print("Parsed and preprocessed", len(title_boost_processed), "documents with separate title/body fields")
print(title_boost_processed[0])

Parsed and preprocessed 1400 documents with separate title/body fields
{'docno': '1', 'title_tokens': ['experiment', 'investig', 'aerodynam', 'wing', 'slipstream'], 'body_tokens': ['experiment', 'investig', 'aerodynam', 'wing', 'slipstream', 'experiment', 'studi', 'wing', 'propel', 'slipstream', 'order', 'determin', 'spanwis', 'distribut', 'lift', 'increas', 'slipstream', 'differ', 'angl', 'attack', 'wing', 'differ', 'free', 'stream', 'slipstream', 'veloc', 'ratio', 'result', 'intend', 'evalu', 'basi', 'differ', 'theoret', 'treatment', 'problem', 'compar', 'span', 'load', 'curv', 'support', 'evid', 'substanti', 'lift', 'increment', 'produc', 'slipstream', 'destal', 'boundari', 'layer', 'control', 'effect', 'integr', 'remain', 'lift', 'increment', 'subtract', 'destal', 'lift', 'agre', 'potenti', 'flow', 'theori', 'empir', 'evalu', 'destal', 'effect', 'specif', 'configur', 'experi']}


In [ ]:
def build_boosted_docs(processed, boost_factor):
    """Repeat each document's title tokens boost_factor times before the body."""
    rows = []
    for d in processed:
        boosted_text = " ".join(d["title_tokens"] * boost_factor + d["body_tokens"])
        rows.append({"docno": d["docno"], "text": boosted_text})
    return pd.DataFrame(rows)

In [ ]:
boost_factors = [1, 2, 3, 5, 8]

boost_tuning_results = []
boost_artifacts = {}

for bf in boost_factors:

    docs_bf = build_boosted_docs(title_boost_processed, bf)

    index_path_bf = f"/content/cranfield_title_boost_{bf}_index"

    indexer_bf = pt.index.IterDictIndexer(
        index_path_bf,
        meta=["docno"],
        text_attrs=["text"],
        overwrite=True
    )

    start = time.perf_counter()

    indexref_bf = indexer_bf.index(docs_bf.to_dict("records"))

    index_time_bf = time.perf_counter() - start

    index_bf = pt.IndexFactory.of(indexref_bf)

    model_bf = pt.terrier.Retriever(
        index_bf,
        wmodel="LemurTF_IDF",
        controls={
            "LemurTF_IDF.k_1": str(best_k1),
            "LemurTF_IDF.b": str(best_b)
        }
    )

    start = time.perf_counter()

    run_bf = model_bf.transform(query)

    search_time_bf = time.perf_counter() - start

    evaluation_bf = pt.Evaluate(
        run_bf,
        qrels,
        metrics=[
            "map",
            "recip_rank",
            "P.5",
            "P.10",
            "ndcg_cut.10"
        ]
    )

    boost_tuning_results.append({
        "boost_factor": bf,
        "MAP": evaluation_bf["map"],
        "MRR": evaluation_bf["recip_rank"],
        "P@5": evaluation_bf["P.5"],
        "P@10": evaluation_bf["P.10"],
        "nDCG@10": evaluation_bf["ndcg_cut.10"],
        "Index Time (s)": index_time_bf,
        "Search Time (s)": search_time_bf
    })

    boost_artifacts[bf] = {
        "docs": docs_bf,
        "index": index_bf,
        "index_time": index_time_bf
    }


boost_tuning_df = pd.DataFrame(boost_tuning_results)

boost_tuning_df = boost_tuning_df.sort_values(
    by="MAP",
    ascending=False
).reset_index(drop=True)

display(boost_tuning_df)

20:24:20.688 [ForkJoinPool-2-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (471) - further warnings are suppressed
20:24:21.918 [ForkJoinPool-2-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
20:24:29.016 [ForkJoinPool-3-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (471) - further warnings are suppressed
20:24:29.923 [ForkJoinPool-3-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
20:24:35.359 [ForkJoinPool-4-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (471) - further warnings are suppressed
20:24:36.268 [ForkJoinPool-4-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
20:24:43.359 [ForkJoinPool-5-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (471) - further warnings are suppressed
2

,boost_factor,MAP,MRR,P@5,P@10,nDCG@10,Index Time (s),Search Time (s)
0,5,0.308062,0.535583,0.314667,0.236889,0.339157,1.229170,4.509360
1,3,0.307700,0.534166,0.312889,0.236000,0.339081,1.256451,6.211111
2,2,0.307290,0.534451,0.309333,0.234667,0.338081,1.276826,4.590395
3,8,0.307272,0.529757,0.313778,0.238222,0.339397,1.332301,5.918084
4,1,0.301681,0.518459,0.304889,0.229778,0.332195,1.637095,6.234498


In [ ]:
best_boost_row = boost_tuning_df.iloc[0]
best_boost_factor = int(best_boost_row["boost_factor"])

print("Best title boost_factor:", best_boost_factor)
print("MAP with best boost:", best_boost_row["MAP"])
print("MAP without title boosting (LemurTF_IDF Tuned):", best["MAP"])

best_boost_docs = boost_artifacts[best_boost_factor]["docs"]
best_boost_index = boost_artifacts[best_boost_factor]["index"]
best_boost_index_time = boost_artifacts[best_boost_factor]["index_time"]

tf_boosted = pt.terrier.Retriever(best_boost_index, wmodel="Tf")

tfidf_boosted = pt.terrier.Retriever(best_boost_index, wmodel="TF_IDF")

tuned_lemur_boosted = pt.terrier.Retriever(
    best_boost_index,
    wmodel="LemurTF_IDF",
    controls={
        "LemurTF_IDF.k_1": str(best_k1),
        "LemurTF_IDF.b": str(best_b)
    }
)

Best title boost_factor: 5
MAP with best boost: 0.3080616251170667
MAP without title boosting (LemurTF_IDF Tuned): 0.301618974194225


### Rocchio (Top-K) on top of the title-boosted index

In [ ]:
start = time.perf_counter()

boosted_rocchio_qe = RocchioQE(
    docs_df=best_boost_docs,
    base_retriever=tuned_lemur_boosted,
    num_fb_docs=int(best_rocchio["num_fb_docs"]),
    alpha=1.0,
    beta=float(best_rocchio["beta"]),
    gamma=0.0,
    expansion_terms=best_expansion_terms,
    top_k=1000
)

boosted_rocchio_build_time = time.perf_counter() - start

start = time.perf_counter()

boosted_rocchio_results = boosted_rocchio_qe.transform(query)

boosted_rocchio_search_time = time.perf_counter() - start

boosted_rocchio_results["qid"] = boosted_rocchio_results["qid"].astype(str)
boosted_rocchio_results["docno"] = boosted_rocchio_results["docno"].astype(str)

boosted_rocchio_eval = pt.Evaluate(
    boosted_rocchio_results,
    qrels,
    metrics=["map", "recip_rank", "P.5", "P.10", "ndcg_cut.10"]
)

print(f"Boosted Rocchio build time: {boosted_rocchio_build_time:.4f}s, "
      f"search time: {boosted_rocchio_search_time:.4f}s")
print(boosted_rocchio_eval)

Boosted Rocchio build time: 0.1516s, search time: 17.9932s
{'map': 0.3445888142629637, 'recip_rank': 0.5564553147150292, 'P.5': 0.3511111111111113, 'P.10': 0.26844444444444454, 'ndcg_cut.10': 0.3746894216001665}


## Grand Final Comparison: Baselines vs. Top-K Rocchio vs. Title Boosting vs. Both Combined

In [ ]:
start = time.perf_counter()
tf_results = tf.transform(query)
tf_search_time = time.perf_counter() - start

start = time.perf_counter()
tfidf_results = tfidf.transform(query)
tfidf_search_time = time.perf_counter() - start

start = time.perf_counter()
tuned_lemur_results = tuned_lemur.transform(query)
tuned_lemur_search_time = time.perf_counter() - start

start = time.perf_counter()
tuned_lemur_boosted_results = tuned_lemur_boosted.transform(query)
tuned_lemur_boosted_search_time = time.perf_counter() - start

rocchio_name = (
    f"Rocchio Top-K (fb_docs={int(best_rocchio['num_fb_docs'])}, "
    f"beta={best_rocchio['beta']}, K={best_expansion_terms})"
)

boosted_lemur_name = f"LemurTF_IDF Tuned + Title Boost (x{best_boost_factor})"

boosted_rocchio_name = (
    f"Rocchio Top-K + Title Boost (x{best_boost_factor}, "
    f"fb_docs={int(best_rocchio['num_fb_docs'])}, beta={best_rocchio['beta']}, "
    f"K={best_expansion_terms})"
)

grand_systems = [
    tf,
    tfidf,
    tuned_lemur,
    best_rocchio_qe,
    tuned_lemur_boosted,
    boosted_rocchio_qe
]

grand_names = [
    "TF",
    "TF_IDF",
    "LemurTF_IDF Tuned",
    rocchio_name,
    boosted_lemur_name,
    boosted_rocchio_name
]

grand_experiment = pt.Experiment(
    grand_systems,
    query,
    qrels,
    eval_metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ],
    names=grand_names
)

grand_timing = pd.DataFrame([
    {"name": "TF", "Index/Build Time (s)": index_time, "Search Time (s)": tf_search_time},
    {"name": "TF_IDF", "Index/Build Time (s)": index_time, "Search Time (s)": tfidf_search_time},
    {"name": "LemurTF_IDF Tuned", "Index/Build Time (s)": index_time, "Search Time (s)": tuned_lemur_search_time},
    {"name": rocchio_name, "Index/Build Time (s)": rocchio_build_time, "Search Time (s)": best_rocchio_search_time},
    {"name": boosted_lemur_name, "Index/Build Time (s)": best_boost_index_time, "Search Time (s)": tuned_lemur_boosted_search_time},
    {"name": boosted_rocchio_name, "Index/Build Time (s)": best_boost_index_time + boosted_rocchio_build_time, "Search Time (s)": boosted_rocchio_search_time},
])

grand_report = grand_experiment.merge(grand_timing, on="name")

display(grand_report)

/usr/local/lib/python3.13/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines do not produce the required outputs for evaluation ({'qid', 'docno', 'score'}):
 - Pipeline #3: Rocchio Top-K (fb_docs=3, beta=1.0, K=None) (<__main__.RocchioQE object at 0x7a7de20a5b20>). Produces []
 - Pipeline #5: Rocchio Top-K + Title Boost (x5, fb_docs=3, beta=1.0, K=None) (<__main__.RocchioQE object at 0x7a7dd6d083c0>). Produces []

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


,name,map,recip_rank,P.5,P.10,ndcg_cut.10,Index/Build Time (s),Search Time (s)
0,TF,0.193439,0.429683,0.194667,0.151556,0.223466,3.859258,4.461647
1,TF_IDF,0.318897,0.554530,0.328889,0.236444,0.346963,3.859258,5.855336
2,LemurTF_IDF Tuned,0.301619,0.518089,0.304889,0.229778,0.332116,3.859258,4.491555
3,"Rocchio Top-K (fb_docs=3, beta=1.0, K=None)",0.340437,0.527942,0.359111,0.270667,0.366833,0.145722,16.599145
4,LemurTF_IDF Tuned + Title Boost (x5),0.308062,0.535583,0.314667,0.236889,0.339157,1.229170,4.462077
5,"Rocchio Top-K + Title Boost (x5, fb_docs=3, be...",0.344589,0.556455,0.351111,0.268444,0.374689,1.380724,17.993224


### Notes for the report

- **Top-K term selection** isolates the effect of pruning the Rocchio feedback centroid down to its `K` highest-weight terms before merging it into the query; the tuning table above shows MAP/nDCG for each `K` (including `"all (no pruning)"` as the baseline) so you can quote the exact gain.
- **Title field boosting** repeats each document's preprocessed title tokens `boost_factor` times before indexing, raising their TF (and hence TF-IDF weight) relative to the body -- a classic, model-agnostic sparse-VSM field-weighting trick. The grid search over `boost_factor` reuses the already-tuned `LemurTF_IDF` `k1`/`b`, so only the effect of boosting itself is being measured.
- The **Grand Final Comparison** shows all four techniques side by side plus their combination (`Rocchio Top-K + Title Boost`), so the report can state clearly whether the two improvements are additive, and quantify each one's individual and combined contribution to MAP/MRR/P@5/P@10/nDCG@10.
- `Index/Build Time (s)` for the boosted Rocchio row is the sum of building the boosted Terrier index and building the Rocchio TF-IDF matrix on top of it, since both are one-off costs paid before any query is answered.
- Everything above remains a **sparse vector space model**: TF, TF-IDF, LemurTF-IDF and Rocchio all operate on term-frequency-based sparse vectors; no probabilistic (BM25/DFR) or dense/neural retrieval model was introduced.